# ITCC508 Lab Exercise PT-M1
## Building and Evaluating a Domain-Specific RAG Chatbot

**Course:** ITC CS08 — Introduction to LLMOps and RAG Concepts
**Topic:** Retrieval-Augmented Generation (RAG) & Hallucination Mitigation
**Student:** Patano Carl Angelo — BSIT 4, Section 402i
**Institution:** Jose Rizal University, Mandaluyong City

**Stack:** `openai/gpt-oss-20b` (Groq) · LangChain · ChromaDB · HuggingFace `all-MiniLM-L6-v2`

> **Note on the generation model.** The handout specifies `llama-3.1-8b-instant`. Groq announced
> the deprecation of that model on 17 June 2026 and decommissioned it on 16 August 2026; requests
> now return HTTP 404 `model_not_found`. Groq's designated replacement, `openai/gpt-oss-20b`, is
> used throughout. Both experimental configurations use the same model, so the comparison in
> Task 3 remains controlled. See Groq's deprecation notice: https://console.groq.com/docs/deprecations

---

## Task 1 — Domain & Dataset Description

**Chosen Domain:** Mandaluyong City tourism, cultural heritage, and visitor services.

**Problem Statement.** Information of this kind — heritage site rules, festival details, office
procedures, permit lead times — is local, procedural, and largely absent from the pretraining
corpora of general-purpose language models. Asked directly, such a model produces fluent but
fabricated answers: invented office hours, invented permit windows. This laboratory builds a
Retrieval-Augmented Generation chatbot that answers strictly from a fixed document corpus, then
measures how strongly retrieval grounding and system-prompt constraints suppress that fabrication.

**Dataset provenance.** The three source documents are a **constructed corpus authored for this
laboratory**, modeled on the structure and register of publicly available municipal tourism
material. They are *not* official publications of the Mandaluyong City government and should not
be cited as factual statements of city policy. A synthetic corpus is appropriate here — and in
some respects preferable — because the object of study is pipeline behaviour rather than the
content itself, and because authoring the corpus gives exact ground truth against which every
retrieved answer can be checked.

**Source files placed in `./my_data/`:**

| # | Filename | Content | Size |
|---|---|---|---|
| 1 | `mandaluyong_city_profile.txt` | City profile, history, festivals, landmarks, transportation | 4,873 chars |
| 2 | `mandaluyong_visitor_faq.txt` | 12-item visitor FAQ — office hours, tour requests, permits, parking | 3,746 chars |
| 3 | `mandaluyong_heritage_guidelines.txt` | Heritage classification, Council rules, nomination and conduct rules | 4,336 chars |

Total corpus: 12,955 characters across 3 plain-text files.

---
## Cell 1 — Environment Setup & Package Installation

**What this cell does, line by line:**

- `!pip install -q ...` — the `!` prefix hands the line to the underlying shell instead of the
  Python interpreter, so packages install into the live Colab runtime. `-q` keeps the log quiet.
- `langchain` — core orchestration library.
- `langchain-community` — community integrations; `DirectoryLoader` and `TextLoader` live here.
- `langchain-classic` — **required in LangChain 1.x.** The `langchain.chains` module used by the
  template was moved out of the core package into this one; without it, Cell 6 raises
  `ModuleNotFoundError: No module named 'langchain.chains'`.
- `langchain-groq` — the `ChatGroq` connector for Groq's inference API.
- `langchain-huggingface` — wraps `sentence-transformers` models in LangChain's `Embeddings`
  interface.
- `langchain-text-splitters` — the standalone package now holding
  `RecursiveCharacterTextSplitter`. Not listed in the handout's Cell 1, but its Cell 3 imports
  from it, so it must be installed explicitly.
- `langchain-chroma` — the maintained Chroma integration.
  `langchain_community.vectorstores.Chroma` is deprecated and emits a removal warning.
- `chromadb` — the vector database engine performing nearest-neighbour search.
- `pypdf` — PDF text extraction, needed if `.pdf` files are placed in `./my_data/`.
- `unstructured` — general-purpose document parser (see the Cell 3 note on why it is not used
  as the loader here).

**Why it is necessary:** RAG is not one library but a pipeline of four independent components
(loader, splitter, embedding model, vector store) plus a generation model. Each package above
supplies exactly one, so none is optional.

In [6]:
# Cell 1: Environment Setup & Package Installation
!pip install -q langchain langchain-community langchain-classic langchain-groq \
                langchain-huggingface langchain-text-splitters langchain-chroma \
                chromadb pypdf unstructured

print("Installation complete. Restart the runtime only if imports below fail.")

Installation complete. Restart the runtime only if imports below fail.


---
## Cell 2 — Secure API Key Configuration

**What this cell does, line by line:**

- `import os` — gives access to `os.environ`. `ChatGroq` reads `GROQ_API_KEY` from it by
  default, so the key never has to be passed as a literal argument.
- `import getpass` — provides a masked input prompt.
- `os.environ.pop("GROQ_API_KEY", None)` — clears any previously stored value before prompting.
  *This replaces the handout's `if "GROQ_API_KEY" not in os.environ:` guard.* That guard skips
  the prompt whenever a value is already present, so an invalid or stale key cannot be corrected
  by re-running the cell — the symptom is a persistent HTTP 401 `invalid_api_key` at the first
  API call, with no indication that the key was never re-read. Clearing first makes the cell
  idempotent.
- `getpass.getpass(...)` — reads the key without echoing it to the screen.
- `.strip()` — removes trailing whitespace or newlines introduced by copy-paste, a common and
  silent cause of 401 errors.

**Why it is necessary:** hard-coding a key as a string literal writes the secret into the
`.ipynb`, which is a submitted, shared, and often version-controlled artifact. Prompting at
runtime keeps the credential out of the saved document entirely.

In [7]:
# Cell 2: Secure API Key Configuration
import os
import getpass

# Clear any stale value, then prompt without echoing plain text into the notebook
os.environ.pop("GROQ_API_KEY", None)
os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ").strip()

print("API key configured for this session.")

Enter your Groq API Key: ··········
API key configured for this session.


---
## Cell 3 — Loading Custom Data & Chunking

**What this cell does, line by line:**

- `from langchain_community.document_loaders import DirectoryLoader, TextLoader` — imports the
  folder walker and the plain-text reader. Each file becomes a `Document` (a `page_content`
  string plus a `metadata` dictionary).
- `from langchain_text_splitters import RecursiveCharacterTextSplitter` — imports the splitter.
- `os.makedirs("my_data", exist_ok=True)` — creates the corpus folder; `exist_ok=True` prevents
  `FileExistsError` on re-run.
- `loader = DirectoryLoader("my_data/", glob="**/*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"}, show_progress=True)` —
  **this differs from the handout, which uses `glob="**/*.*"` with the default loader class.**
  The default routes every file through the `unstructured` parser, which returned zero documents
  for these `.txt` files while reporting no error; the failure only surfaced two cells later as
  `ValueError: Expected Embeddings to be non-empty list ... got []` from Chroma. Passing
  `loader_cls=TextLoader` reads the files directly and resolved it. `glob="**/*.txt"` narrows the
  match accordingly, and `encoding="utf-8"` makes decoding explicit rather than platform-dependent.
- `raw_documents = loader.load()` — performs the read, one `Document` per file, each tagged with
  a `source` metadata field holding its path.
- `RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)` — caps each chunk near 500
  characters. "Recursive" means it tries a descending priority of separators (paragraph, line,
  sentence, word), falling back to a harder cut only when a chunk still exceeds the limit, so
  splits land on natural boundaries. The 50-character overlap repeats the tail of each chunk at
  the head of the next, so a fact straddling a boundary survives in at least one chunk.
- `documents = text_splitter.split_documents(raw_documents)` — applies the split and propagates
  `metadata` onto every child chunk, which is what makes source attribution possible in Cell 7.
- `print(f"...")` — the first debugging checkpoint. A count of `0` here means every later cell
  will fail or produce empty output.

**Why it is necessary:** an embedding model emits one fixed-length vector per input, so encoding
a whole 4,000-character document would average many unrelated facts into a single vector and
destroy retrieval precision. Chunking gives each vector a narrow, single-topic meaning.

In [8]:
# Cell 3: Loading Custom Data & Chunking
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create target data directory
os.makedirs("my_data", exist_ok=True)

# Load all documents from the directory.
# NOTE: loader_cls=TextLoader replaces the handout's default (unstructured), which
# silently returned 0 documents for these .txt files. See the markdown above.
loader = DirectoryLoader(
    "my_data/",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True,
)
raw_documents = loader.load()

# Split documents into smaller semantic chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
documents = text_splitter.split_documents(raw_documents)

print(f"Loaded {len(raw_documents)} raw document(s) and split into {len(documents)} chunks.")

100%|██████████| 3/3 [00:00<00:00, 3486.54it/s]

Loaded 3 raw document(s) and split into 37 chunks.


### Chunk inspection (supplementary verification)

Not part of the handout. Included as a debugging checkpoint after the silent loader failure
described above: it confirms the loader actually read text, reports the realised chunk statistics,
and prints one representative chunk before anything is embedded. The explicit empty-corpus branch
turns a downstream `ValueError` from Chroma into a diagnosable message at the point of failure.

In [9]:
# Verification: inspect the corpus before embedding
print("Source files detected:")
for d in raw_documents:
    print("  -", d.metadata.get("source", "Unknown"), f"({len(d.page_content)} chars)")

if not documents:
    print("\n*** CORPUS EMPTY — upload the .txt files into my_data/ and re-run Cell 3. ***")
else:
    print(f"\nTotal chunks: {len(documents)}")
    print(f"Average chunk length: {sum(len(d.page_content) for d in documents) / len(documents):.0f} chars")
    print("\n--- SAMPLE CHUNK ---")
    print(documents[min(5, len(documents) - 1)].page_content)

Source files detected:
  - my_data/mandaluyong_visitor_faq.txt (3746 chars)
  - my_data/mandaluyong_city_profile.txt (4873 chars)
  - my_data/mandaluyong_heritage_guidelines.txt (4336 chars)

Total chunks: 37
Average chunk length: 355 chars

--- SAMPLE CHUNK ---
Q6: What is the maximum group size for a guided heritage tour?
A6: The standard maximum group size is forty (40) participants per tour schedule. Groups larger
than forty will be divided into separate batches assigned to different time slots.




---
## Cell 4 — Embedding Model & Vector DB Indexing

**What this cell does, line by line:**

- `from langchain_huggingface import HuggingFaceEmbeddings` — imports the wrapper that exposes a
  local `sentence-transformers` model through LangChain's `Embeddings` interface, so the vector
  store can call it without knowing which model is behind it.
- `from langchain_chroma import Chroma` — imports the maintained Chroma integration. The handout's
  `langchain_community.vectorstores.Chroma` still works but is deprecated and emits a removal
  warning on import.
- `embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")` — downloads (once) and loads
  a six-layer sentence-transformer that maps any text to a 384-dimensional dense vector. Texts with
  similar meaning land close together in that space, which is the property retrieval relies on.
  It runs on the notebook CPU, so embedding is free and no document text leaves the runtime.
- `vectorstore = Chroma.from_documents(documents=documents, embedding=embeddings,
  collection_name="mandaluyong_rag")` — embeds every chunk from Cell 3, then writes each vector,
  its original text, and its `metadata` (including `source`) into an in-memory ChromaDB
  collection. `collection_name` is an addition to the handout: without it, re-running the cell
  creates a second anonymous collection and duplicates every chunk in the index.
- `retriever = vectorstore.as_retriever(search_kwargs={"k": 3})` — wraps the store as a
  `Retriever` runnable. At query time it embeds the question with the *same* model, runs a
  nearest-neighbour search, and returns the three closest chunks. Note that `k=3` is always
  satisfied — there is no relevance threshold, so an off-topic question still retrieves three
  chunks, just poor ones. This matters for Tasks 2 and 3.
- `print(f"Indexed ...")` — a second debugging checkpoint confirming the count that reached the
  index equals the count produced by the splitter.

**Why it is necessary:** this cell is the *retrieval* half of RAG. The embedding model turns text
into geometry so that similarity can be computed numerically, and the vector store makes that
search fast and returns the source text alongside each hit. Without it the LLM would have no way
to be handed the passages relevant to a question; with it, every answer can be traced back to the
exact chunks that were retrieved.


In [10]:
# Cell 4: Embedding Model & Vector DB Indexing
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Initialize open-source embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Store embeddings into Chroma vector database.
# NOTE: a named collection prevents duplicate accumulation across re-runs.
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="mandaluyong_rag",
)

# Set vectorstore as a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Indexed {len(documents)} chunks into ChromaDB. Retriever configured with k=3.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexed 37 chunks into ChromaDB. Retriever configured with k=3.






---
## Cell 5 — Model Initialization and Domain System Prompt (BASELINE CONFIGURATION)

**What this cell does, line by line:**

- `from langchain_groq import ChatGroq` — imports the chat-model class backed by Groq's LPU
  inference endpoint.
- `from langchain_core.prompts import ChatPromptTemplate` — imports the template object that
  formats role-tagged messages and exposes `{placeholders}` for runtime substitution.
- `llm = ChatGroq(model_name="openai/gpt-oss-20b", temperature=0)` — **the handout specifies
  `llama-3.1-8b-instant`, which Groq deprecated on 17 June 2026 and decommissioned on 16 August
  2026; it now returns HTTP 404 `model_not_found`.** `openai/gpt-oss-20b` is Groq's designated
  replacement and is used in both configurations, preserving the controlled comparison.
  `temperature=0` makes decoding effectively greedy: the highest-probability token is selected at
  each step rather than sampled. This is the primary determinism knob and makes the experiment
  reproducible.
- `system_prompt = (...)` — a multi-line string built by implicit adjacent-string concatenation,
  carrying three distinct instructions:
  1. a **role assignment** narrowing the model's persona to a domain assistant;
  2. a **grounding constraint** — answer using *only* the provided context;
  3. a **refusal clause** — an explicit verbatim fallback sentence for when the context does not
     contain the answer. A model with no sanctioned way to say "I don't know" will tend to invent
     something instead.

  The trailing `"Context:\n{context}"` is the injection point for retrieved chunks.
- `ChatPromptTemplate.from_messages([...])` — assembles the template: a `"system"` turn holding
  constraints and context, a `"human"` turn holding `{input}`. Keeping context in the system turn
  and the question in the human turn preserves the instruction hierarchy the model was tuned on.

**Why it is necessary:** retrieval alone does not guarantee grounding — the model can still ignore
the retrieved passages and answer from pretraining knowledge. The system prompt is the policy
layer that forbids this, and Task 3 measures how much of the grounding it actually contributes.

In [11]:
# Cell 5: Model Initialization and Domain System Prompt
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# Initialize the SLM
# NOTE: llama-3.1-8b-instant (per handout) was decommissioned by Groq on 2026-08-16.
# openai/gpt-oss-20b is Groq's designated replacement.
llm = ChatGroq(
    model_name="openai/gpt-oss-20b",
    temperature=0
)

# Custom domain system prompt
system_prompt = (
    "You are a specialized AI assistant for the user's uploaded domain.\n"
    "Answer questions strictly using ONLY the provided context below.\n"
    "If the answer cannot be found in the context, reply: 'I cannot answer based on the provided domain data.'\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

print("Baseline configuration ready: temperature=0, fallback rule ENABLED.")

Baseline configuration ready: temperature=0, fallback rule ENABLED.


---
## Cell 6 — Pipeline Assembly

**What this cell does, line by line:**

- `from langchain_classic.chains import create_retrieval_chain` — imports the factory joining a
  retriever to a document-consuming chain. **The handout uses `from langchain.chains import ...`,
  which raises `ModuleNotFoundError` on LangChain 1.x:** the `chains` module was moved out of the
  core package into `langchain-classic` (installed in Cell 1). The classes are unchanged.
- `from langchain_classic.chains.combine_documents import create_stuff_documents_chain` — imports
  the factory for the "stuff" strategy, the simplest document-combination strategy.
- `combine_docs_chain = create_stuff_documents_chain(llm, prompt)` — builds a runnable that
  accepts `Document` objects, concatenates ("stuffs") their `page_content` into the `{context}`
  placeholder, and sends the filled prompt to the LLM. Stuffing suits this corpus because
  *k*=3 × ~355 characters ≈ 1,065 characters, far inside the context window; a larger corpus would
  need map-reduce or refine.
- `rag_chain = create_retrieval_chain(retriever, combine_docs_chain)` — produces the end-to-end
  chain. On `.invoke({"input": q})` it executes: embed `q` → similarity search → take top-3 →
  stuff into prompt → call the LLM → return a dictionary with `input`, `context`, and `answer`.

**Why it is necessary:** this is the seam between retrieval and generation. Returning `context`
alongside `answer` is what makes the system auditable — every claim traces to the chunk that
produced it, which is the operational basis for detecting hallucination in Task 3.

In [12]:
# Cell 6: Pipeline Assembly
# NOTE: langchain_classic replaces langchain.chains, which was removed from the
# core package in LangChain 1.x.
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Combine prompt and LLM to process context
combine_docs_chain = create_stuff_documents_chain(llm, prompt)

# Assemble full retrieval-augmented generation chain
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

print("RAG chain assembled.")

RAG chain assembled.


---
## Cell 7 — Testing Your Custom Domain Chatbot

**What this cell does, line by line:**

- `user_query = "..."` — the question, stored in a variable so the identical string can be reused
  across configurations.
- `response = rag_chain.invoke({"input": user_query})` — runs the full pipeline. The key
  `"input"` must match the placeholder declared in the human message of Cell 5.
- `print(response["answer"])` — prints the generated text.
- `for i, doc in enumerate(response["context"]):` — iterates the retrieved chunks; `enumerate`
  supplies the index for the `Chunk {i+1}` label.
- `doc.metadata.get("source", "Unknown")` — reads the source path recorded by the loader in
  Cell 3 and carried through the splitter. `.get()` with a default avoids `KeyError` on a chunk
  missing the field.

**Why it is necessary:** printing retrieved chunks alongside the answer turns the chatbot from a
black box into an inspectable system. A fact appearing in the answer but in none of the printed
chunks is, by definition, a hallucination.

In [13]:
# Cell 7: Testing Your Custom Domain Chatbot

# Test Case 1: In-Domain Query
user_query = "Summarize the key information found in the documents."
response = rag_chain.invoke({"input": user_query})

print("--- DOMAIN QUERY ANSWER ---")
print(response["answer"])

print("\n--- RETRIEVED SOURCE CHUNKS ---")
for i, doc in enumerate(response["context"]):
    print(f"Chunk {i+1} Source:", doc.metadata.get("source", "Unknown"))

--- DOMAIN QUERY ANSWER ---
**Key Information from the Documents**

1. **Nomination Requirements for Cultural Properties**  
   - **Location Details**: Exact address, lot number, and title reference of the property.  
   - **Narrative Statement**: A concise description of the property’s significance (≤ 1,000 words).  
   - **Photographic Evidence**: Minimum of five photographs taken within the last 12 months.  
   - **Age Documentation**: Proof of the property’s age (construction record, tax declaration, or sworn statement from a long‑time resident).  

2. **Inventory Update Procedure (Section 4)**  
   - The city inventory of cultural properties is reviewed annually.  
   - Any resident, organization, or academic institution may nominate a property by submitting a nomination form to the Tourism Office.  
   - A complete nomination must include the items listed above (the full list is implied but not fully shown in the excerpt).  

3. **Researcher Access (Section 6)**  
   - Researcher

### Additional in-domain queries (baseline)

Three further in-domain questions establish that the baseline retrieves and answers correctly
before it is stress-tested. The helper below wraps the invoke-and-print pattern so the same
procedure is reused for every remaining test, including a preview of each retrieved chunk's text
rather than only its filename.

In [14]:
def ask(chain, query, label=""):
    """Invoke a RAG chain and print the answer together with its retrieved sources."""
    resp = chain.invoke({"input": query})
    print("=" * 78)
    if label:
        print(f"[{label}]")
    print("QUERY :", query)
    print("-" * 78)
    print("ANSWER:", resp["answer"])
    print("-" * 78)
    print("RETRIEVED CHUNKS:")
    for i, d in enumerate(resp["context"]):
        preview = d.page_content[:110].replace("\n", " ")
        print(f"  [{i+1}] {d.metadata.get('source', 'Unknown')} :: {preview}...")
    print("=" * 78, "\n")
    return resp


in_domain_queries = [
    "What does the acronym SUMAKAH stand for?",
    "How many working days in advance must a guided heritage walking tour be requested, and what is the maximum group size?",
    "How many members compose the City Heritage Council and what constitutes a quorum?",
]

baseline_results = [ask(rag_chain, q, "BASELINE  temp=0  fallback=ON") for q in in_domain_queries]

[BASELINE  temp=0  fallback=ON]
QUERY : What does the acronym SUMAKAH stand for?
------------------------------------------------------------------------------
ANSWER: SUMAKAH stands for:

- **SUMAsayaw** – dancing  
- **MAsayahin** – cheerful  
- **KAlakalan** – commerce  
- **Hardin** – garden
------------------------------------------------------------------------------
RETRIEVED CHUNKS:
  [1] my_data/mandaluyong_city_profile.txt :: 3.1 Sumakah Festival The Sumakah Festival is the flagship cultural celebration of Mandaluyong City, held annua...
  [2] my_data/mandaluyong_heritage_guidelines.txt :: Category C — Cultural Landscape or Public Space An open area, park, plaza, or rotunda associated with a docume...
  [3] my_data/mandaluyong_visitor_faq.txt :: Q9: Does the city offer souvenir or local products? A9: Locally produced items, including garments and handicr...

[BASELINE  temp=0  fallback=ON]
QUERY : How many working days in advance must a guided heritage walking tour be request

---
# Task 2 — Out-of-Domain Fallback Test

**Procedure.** A question is posed whose subject matter is entirely absent from the corpus. The
retriever will still return three chunks: vector search has no relevance floor and always returns
its top-*k* nearest neighbours no matter how poor the match. The correct behaviour is therefore
**not** an empty retrieval, but the model recognising that the retrieved passages do not contain
the answer and emitting the exact fallback sentence.

**Out-of-domain query:** *"What is the average distance from Earth to Mars, and what propulsion
system did the Apollo 11 spacecraft use?"*

**Expected output:** `I cannot answer based on the provided domain data.`

In [15]:
# Task 2: Out-of-Domain Fallback Test
OUT_OF_DOMAIN_QUERY = (
    "What is the average distance from Earth to Mars, and what propulsion system "
    "did the Apollo 11 spacecraft use?"
)

baseline_ood = ask(rag_chain, OUT_OF_DOMAIN_QUERY, "TASK 2 — BASELINE  temp=0  fallback=ON")

# Programmatic verification of the fallback string
FALLBACK = "I cannot answer based on the provided domain data."
print("Fallback message correctly returned:", FALLBACK.lower() in baseline_ood["answer"].lower())

[TASK 2 — BASELINE  temp=0  fallback=ON]
QUERY : What is the average distance from Earth to Mars, and what propulsion system did the Apollo 11 spacecraft use?
------------------------------------------------------------------------------
ANSWER: I cannot answer based on the provided domain data.
------------------------------------------------------------------------------
RETRIEVED CHUNKS:
  [1] my_data/mandaluyong_visitor_faq.txt :: Q6: What is the maximum group size for a guided heritage tour? A6: The standard maximum group size is forty (4...
  [2] my_data/mandaluyong_visitor_faq.txt :: Q2: What are the office hours of the City Tourism Office? A2: The office is open Monday to Friday, from 8:00 A...
  [3] my_data/mandaluyong_heritage_guidelines.txt :: (a) the exact location and, if available, the lot and title reference of the property; (b) a narrative stateme...

Fallback message correctly returned: True


### Task 2 — Observed Result

| Field | Value |
|---|---|
| Query | What is the average distance from Earth to Mars, and what propulsion system did the Apollo 11 spacecraft use? |
| Expected | `I cannot answer based on the provided domain data.` |
| Actual output | `I cannot answer based on the provided domain data.` |
| Fallback triggered | **True** (programmatic string match) |
| Chunks retrieved | 3 — two from `mandaluyong_visitor_faq.txt` (Q6 group size, Q2 office hours), one from `mandaluyong_heritage_guidelines.txt` (nomination requirements); all irrelevant to the query |

The retrieval was not empty. The refusal was a judgement by the model about the *sufficiency* of
the retrieved context, not a mechanical consequence of finding nothing.

---
# Task 3 — Hallucination Stress-Test

## Step 1 — Change Parameters (UNGROUNDED CONFIGURATION)

Two modifications are made relative to Cell 5, and only these two:

1. `temperature` is raised from `0` to `1.0`. Decoding switches from greedy selection to sampling
   from the token probability distribution, making lower-probability continuations reachable.
2. The refusal clause — the line beginning *"If the answer cannot be found in the context,
   reply..."* — is **deleted** from the system prompt.

The corpus, chunking parameters, embedding model, retriever, *k*, and generation model are all
held constant, so any behavioural difference is attributable to the two changes above.

**A design limitation worth stating in advance.** Step 1 removes only the sentence prescribing the
*verbatim* refusal string. The preceding constraint — *"Answer questions strictly using ONLY the
provided context below"* — remains in the ungrounded prompt. The two lines are therefore not
independent: what is removed is the refusal's prescribed wording, not the grounding instruction
itself. The results below turn on exactly this point.

In [16]:
# Task 3 Step 1: Ungrounded configuration — temperature=1.0, fallback rule REMOVED

llm_ungrounded = ChatGroq(
    model_name="openai/gpt-oss-20b",   # same model as baseline — comparison stays controlled
    temperature=1.0                    # CHANGED: 0 -> 1.0
)

system_prompt_ungrounded = (
    "You are a specialized AI assistant for the user's uploaded domain.\n"
    "Answer questions strictly using ONLY the provided context below.\n"
    # DELETED: the "If the answer cannot be found in the context, reply: ..." fallback line
    "\n"
    "Context:\n{context}"
)

prompt_ungrounded = ChatPromptTemplate.from_messages([
    ("system", system_prompt_ungrounded),
    ("human", "{input}"),
])

combine_docs_chain_ungrounded = create_stuff_documents_chain(llm_ungrounded, prompt_ungrounded)
rag_chain_ungrounded = create_retrieval_chain(retriever, combine_docs_chain_ungrounded)

print("Ungrounded configuration ready: temperature=1.0, fallback rule REMOVED.")

Ungrounded configuration ready: temperature=1.0, fallback rule REMOVED.


## Step 2 — Re-run the Same Out-of-Domain Query

The identical string from Task 2 is reused. It is sampled three times: at `temperature=1.0` the
output is stochastic, so a single sample would not be adequate evidence of the configuration's
behaviour. Run-to-run variance is itself part of the finding.

In [17]:
# Task 3 Step 2: Re-run the exact same out-of-domain query under the ungrounded configuration
ungrounded_runs = []
for trial in range(1, 4):
    r = ask(rag_chain_ungrounded, OUT_OF_DOMAIN_QUERY, f"TASK 3 — UNGROUNDED  temp=1.0  fallback=OFF  (run {trial}/3)")
    ungrounded_runs.append(r["answer"])

print("Fallback phrase present in any ungrounded run:",
      any(FALLBACK.lower() in a.lower() for a in ungrounded_runs))

[TASK 3 — UNGROUNDED  temp=1.0  fallback=OFF  (run 1/3)]
QUERY : What is the average distance from Earth to Mars, and what propulsion system did the Apollo 11 spacecraft use?
------------------------------------------------------------------------------
ANSWER: I’m sorry, but I don’t have that information in the context provided.
------------------------------------------------------------------------------
RETRIEVED CHUNKS:
  [1] my_data/mandaluyong_visitor_faq.txt :: Q6: What is the maximum group size for a guided heritage tour? A6: The standard maximum group size is forty (4...
  [2] my_data/mandaluyong_visitor_faq.txt :: Q2: What are the office hours of the City Tourism Office? A2: The office is open Monday to Friday, from 8:00 A...
  [3] my_data/mandaluyong_heritage_guidelines.txt :: (a) the exact location and, if available, the lot and title reference of the property; (b) a narrative stateme...

[TASK 3 — UNGROUNDED  temp=1.0  fallback=OFF  (run 2/3)]
QUERY : What is the average 

## Step 2b — Side-by-Side Summary

A compact printout of the baseline answer against the three ungrounded samples, for direct
transcription into Table II of the IEEE report.

In [18]:
print("#" * 78)
print("SIDE-BY-SIDE COMPARISON — identical out-of-domain query")
print("#" * 78)
print("\nQUERY:", OUT_OF_DOMAIN_QUERY)

print("\n>>> CONFIG A — BASELINE (temperature=0, fallback rule ENABLED)")
print(baseline_ood["answer"])

for i, a in enumerate(ungrounded_runs, 1):
    print(f"\n>>> CONFIG B — UNGROUNDED run {i} (temperature=1.0, fallback rule REMOVED)")
    print(a)
print("\n" + "#" * 78)

##############################################################################
SIDE-BY-SIDE COMPARISON — identical out-of-domain query
##############################################################################

QUERY: What is the average distance from Earth to Mars, and what propulsion system did the Apollo 11 spacecraft use?

>>> CONFIG A — BASELINE (temperature=0, fallback rule ENABLED)
I cannot answer based on the provided domain data.

>>> CONFIG B — UNGROUNDED run 1 (temperature=1.0, fallback rule REMOVED)
I’m sorry, but I don’t have that information in the context provided.

>>> CONFIG B — UNGROUNDED run 2 (temperature=1.0, fallback rule REMOVED)
I’m sorry, but I don’t have that information available in the provided context.

>>> CONFIG B — UNGROUNDED run 3 (temperature=1.0, fallback rule REMOVED)
I’m sorry, but I don’t have that information in the context provided.

##############################################################################


## Step 2c — Cross-check on an In-Domain Query

An additional control: the same factual in-domain question is put to both configurations. This
separates two distinct failure modes — *refusing correctly on out-of-domain input* versus
*staying accurate on in-domain input*. A configuration can pass one and fail the other.

In [19]:
CONTROL_QUERY = "How many working days in advance must a guided heritage walking tour be requested, and what is the maximum group size?"

print("### CONFIG A — BASELINE ###")
_ = ask(rag_chain, CONTROL_QUERY, "BASELINE  temp=0  fallback=ON")

print("### CONFIG B — UNGROUNDED ###")
_ = ask(rag_chain_ungrounded, CONTROL_QUERY, "UNGROUNDED  temp=1.0  fallback=OFF")

print("Ground truth from mandaluyong_visitor_faq.txt (Q5, Q6): ten (10) working days; maximum forty (40) participants.")

### CONFIG A — BASELINE ###
[BASELINE  temp=0  fallback=ON]
QUERY : How many working days in advance must a guided heritage walking tour be requested, and what is the maximum group size?
------------------------------------------------------------------------------
ANSWER: You must submit the request at least **10 working days** before the intended date, and the **maximum group size** for a guided heritage walking tour is **40 participants**.
------------------------------------------------------------------------------
RETRIEVED CHUNKS:
  [1] my_data/mandaluyong_visitor_faq.txt :: Q6: What is the maximum group size for a guided heritage tour? A6: The standard maximum group size is forty (4...
  [2] my_data/mandaluyong_visitor_faq.txt :: Q5: How do I request a guided heritage walking tour? A5: Guided heritage walking tours may be requested by sub...
  [3] my_data/mandaluyong_visitor_faq.txt :: Q2: What are the office hours of the City Tourism Office? A2: The office is open Monday to Fr

### Step 3 — Observe & Compare

#### 3.1 Did the model remain grounded in the retrieved context?

**Baseline (temperature = 0, fallback enabled).** Yes. The system returned the prescribed sentence verbatim — *"I cannot answer based on the provided domain data."* — and the programmatic check confirmed the match (`True`). Retrieval was not empty: three chunks were returned, two from the visitor FAQ and one from the heritage guidelines. The refusal was therefore a judgement about the sufficiency of the retrieved context, not a consequence of failed retrieval. On the three in-domain queries the baseline answered correctly in every case, with each answer traceable to a printed chunk.

**Ungrounded (temperature = 1.0, fallback removed).** Also grounded — the model refused in all three samples, but never using the prescribed wording. The observed outputs were:

Run 1: "I'm sorry, but I don't have that information in the context provided."

Run 2: "I'm sorry, but I don't have that information available in the provided context."

Run 3: "I'm sorry, but I don't have that information in the context provided."

All three name the provided context as the reason for declining, so all three read as grounding decisions rather than generic refusals. The verification check reported `False`, but not because the model hallucinated: it declined in its own words rather than the prescribed string. The predicted failure mode did not occur.

#### 3.2 Did it become less reliable?

Barely, and only in surface wording. Runs 1 and 3 were byte-identical, and run 2 differed from them by a single inserted word ("available") and a reordering of the closing phrase. Every sample was the same semantic act — declining, with the provided context named as the reason — and none introduced content from outside the corpus. Raising temperature to 1.0 therefore did not meaningfully widen the output distribution here: the refusal was not a marginal high-probability outcome that sampling could dislodge, but a decision the model reached robustly across the sampled distribution. Had a fabricated continuation been anywhere near competitive in probability, three samples at temperature 1.0 would have been a fair chance to surface one.

On the in-domain control query both configurations returned the correct values (10 working days; maximum 40 participants) from the same three retrieved chunks, so accuracy on answerable questions was unaffected by either change.

#### 3.3 Was unsupported information introduced?

None, in any run. Every claim across all four out-of-domain responses was a statement about the absence of information, which correctly characterises the retrieved context. No figure for Earth–Mars distance and no description of Apollo 11 propulsion appeared in any output, despite both being squarely within the model's pretraining knowledge.

The distinction this experiment was designed to expose still holds and is worth stating even though the fabrication did not occur: **factual correctness and grounding are not the same property.** Had Config B returned an accurate Earth–Mars figure, the system would still have failed its grounding contract, because that figure appears in none of the retrieved chunks. The mechanism producing a correct answer from parametric memory is the same one that produces a confident wrong answer on a topic the model does not actually know — and from the user's side the two are indistinguishable.

#### 3.4 Did it refuse to answer?

| Configuration | Refusals | Verbatim fallback string |
|---|---|---|
| A — baseline (t=0, fallback ON) | 1 / 1 | Yes |
| B — ungrounded (t=1.0, fallback OFF) | 3 / 3 | No — self-worded in all three |

Total refusal rate on out-of-domain input: **4 / 4.** Hallucination rate: **0 / 4.**

#### 3.5 Explanation of the observed differences

The experiment did not reproduce the expected hallucination, and the reason is instructive rather than a defect in execution.

**The grounding instruction was never actually removed.** Step 1 deletes the sentence prescribing the verbatim refusal string, but leaves intact the preceding constraint: *"Answer questions strictly using ONLY the provided context below."* That line alone was sufficient to keep the model grounded. What the modification removed was the *output format* of the refusal — which is exactly what the results show, since the model continued to refuse while abandoning the prescribed wording. The two prompt lines are not independent variables, and the design conflates them.

**The wording of the refusals is consistent with that reading, but does not prove it.** Every sample attributed the gap to the provided context, which is what would be expected if the surviving grounding constraint were doing the work. A model declining from general helpfulness training could phrase itself the same way, however, so the outputs alone cannot separate the two mechanisms — only the ablation described below can.

**Model substitution limits generalisation.** The handout specifies `llama-3.1-8b-instant`, which Groq decommissioned on 16 August 2026; `openai/gpt-oss-20b` was substituted. The replacement is a newer and larger model whose instruction-following and refusal behaviour is plausibly stronger than the 8B Llama the exercise was designed around. The negative result should therefore be read as specific to this model, not as evidence that prompt constraints are dispensable in general.

**A better-designed follow-up.** To isolate which instruction actually performs the grounding work, the two prompt lines should be ablated separately across four conditions: both lines present; refusal clause removed only; grounding constraint removed only; both removed. Each condition sampled several times, at both temperature settings. The present two-condition design cannot distinguish "the refusal clause was unnecessary" from "the grounding constraint was doing all the work," and the four-condition design would resolve exactly that.

---
## Consolidated Results Table (source for Table II of the IEEE report)

| # | Query | Type | Config | Expected Response | Actual Model Output | Grounded? |
|---|---|---|---|---|---|---|
| 1 | What does the acronym SUMAKAH stand for? | In-domain | A | SUMAsayaw, MAsayahin, KAlakalan, Hardin | SUMAsayaw – dancing; MAsayahin – cheerful; KAlakalan – commerce; Hardin – garden | Yes |
| 2 | Tour lead time and max group size | In-domain | A | 10 working days; 40 participants | At least ten (10) working days in advance; maximum group size 40 | Yes |
| 3 | Heritage Council composition and quorum | In-domain | A | 7 members; quorum of 4 | Seven (7) members; quorum of four (4) | Yes |
| 4 | Earth–Mars distance; Apollo 11 propulsion | Out-of-domain | A | `I cannot answer based on the provided domain data.` | `I cannot answer based on the provided domain data.` | Yes — verbatim |
| 5 | Same as #4 (run 1) | Out-of-domain | B | No prescribed refusal available |"I'm sorry, but I don't have that information in the context provided." | Yes — self-worded |
| 6 | Same as #4 (run 2) | Out-of-domain | B | — | "I'm sorry, but I don't have that information available in the provided context."| Yes — self-worded |
| 7 | Same as #4 (run 3) | Out-of-domain | B | — | "I'm sorry, but I don't have that information in the context provided." (identical to run 1)| Yes — self-worded |
| 8 | Tour lead time and max group size (control) | In-domain | B | 10 working days; 40 participants | At least 10 working days in advance; maximum group size 40 | Yes |

### Implementation Parameters (source for Section II of the IEEE report)

| Parameter | Value | Rationale |
|---|---|---|
| Document loader | `DirectoryLoader` + `TextLoader` | `unstructured` (handout default) returned 0 documents for `.txt` |
| Chunk size | 500 characters | One topic per vector; preserves retrieval precision |
| Chunk overlap | 50 characters (10%) | Prevents loss of facts split across a boundary |
| Realised chunk count | 37 chunks (avg. 355 chars) | From 3 files totalling 12,955 characters |
| Embedding model | `all-MiniLM-L6-v2` (384-dim) | Local, free, strong for short passages |
| Vector store | ChromaDB (in-memory) | Session-scoped lab corpus; no persistence required |
| Retrieval top-*k* | 3 | Balances recall against context dilution |
| Generation model | `openai/gpt-oss-20b` via Groq | Substituted for decommissioned `llama-3.1-8b-instant` |
| Temperature (Config A) | 0 | Deterministic, reproducible baseline |
| Temperature (Config B) | 1.0 | Stochastic decoding for the stress-test |
| Fallback rule (A / B) | Enabled / Removed | Independent variable of the stress-test |
| Combination strategy | `stuff` | ~1,065 chars of context fits the window comfortably |

### Documented Deviations from the Handout

| # | Handout specifies | Used instead | Reason |
|---|---|---|---|
| 1 | Cell 1 package list | Added `langchain-text-splitters`, `langchain-chroma`, `langchain-classic` | Handout imports from packages it never installs |
| 2 | `if "GROQ_API_KEY" not in os.environ` | `os.environ.pop(...)` then prompt | Guard prevents correcting a stale/invalid key; causes persistent HTTP 401 |
| 3 | `DirectoryLoader(glob="**/*.*")` default parser | `loader_cls=TextLoader`, `glob="**/*.txt"` | `unstructured` returned 0 documents silently |
| 4 | `langchain_community.vectorstores.Chroma` | `langchain_chroma.Chroma` | Former is deprecated, emits removal warning |
| 5 | `from langchain.chains import ...` | `from langchain_classic.chains import ...` | Module moved out of core package in LangChain 1.x |
| 6 | `llama-3.1-8b-instant` | `openai/gpt-oss-20b` | Decommissioned by Groq 2026-08-16; returns HTTP 404 |

### References

1. LangChain, *Retrieval-augmented generation (RAG) — conceptual guide.* https://python.langchain.com/docs/concepts/rag/
2. LangChain, *DirectoryLoader and TextLoader — API reference.* https://python.langchain.com/api_reference/community/document_loaders/
3. LangChain, *langchain-classic migration notes (v1.0).* https://python.langchain.com/docs/versions/v1/
4. N. Reimers and I. Gurevych, "Sentence-BERT: Sentence embeddings using Siamese BERT-networks," *Proc. EMNLP-IJCNLP*, 2019, pp. 3982–3992.
5. Hugging Face, *sentence-transformers/all-MiniLM-L6-v2 — model card.* https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
6. Chroma, *Chroma documentation.* https://docs.trychroma.com/
7. Groq, *Model deprecations.* https://console.groq.com/docs/deprecations
8. Groq, *GroqCloud documentation — supported models and parameters.* https://console.groq.com/docs/
9. P. Lewis et al., "Retrieval-augmented generation for knowledge-intensive NLP tasks," *Proc. NeurIPS*, 2020, pp. 9459–9474.
10. Z. Ji et al., "Survey of hallucination in natural language generation," *ACM Computing Surveys*, vol. 55, no. 12, pp. 1–38, 2023.
11. Python Software Foundation, *getpass — portable password input.* https://docs.python.org/3/library/getpass.html